# 02 — Unit Conversion in pytrist

Tristan-MP V2 uses an internal (code) unit system:

| Quantity | Code unit |
|----------|-----------|
| Length   | grid cells |
| Time     | $1/\omega_{pe}$ (inverse electron plasma frequency) |
| Speed    | $c$ (speed of light) |
| B field  | normalised to $B_0$ |
| E field  | normalised to $B_0$ |

For magnetised reconnection physics, it is more natural to work in **ion units** set by the background magnetic field $B_y$:

| Quantity | Ion unit | Definition |
|----------|----------|------------|
| Length   | $d_i$ (ion inertial length) | $d_i = d_e\,\sqrt{m_i/m_e}$ |
| Time     | $\Omega_{ci}^{-1}$ (ion cyclotron period) | $\Omega_{ci} = \sqrt{\sigma}\,\omega_{pe}/( m_i/m_e)$ |
| Speed    | $v_{Ai}$ (ion Alfvén speed) | $v_{Ai} = c\,\sqrt{\sigma/(m_i/m_e)}$ |
| B field  | $B_0$ | unchanged (dimensionless) |
| E field  | $E_0 = B_0\,(v_{Ai}/c)$ | MHD electric field scale |

where $\sigma = \omega_{ce}^2/\omega_{pe}^2$ is the magnetisation parameter.

This notebook demonstrates every conversion using `UnitConverter`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pytrist.units import UnitConverter

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

## 1  Building a UnitConverter

You can construct `UnitConverter` directly from four numbers found in the Tristan params file,
or retrieve it automatically from a `Simulation` object (which reads the params file for you).

Here we use parameters from a typical reconnection run:

In [ ]:
# Typical Tristan-V2 reconnection parameters
c_omp      = 10.0    # electron skin depth de in cells
sigma      = 0.1     # magnetisation σ = ωce²/ωpe²
mass_ratio = 100.0   # mi/me
CC         = 0.45    # speed of light in code units

uc = UnitConverter(c_omp=c_omp, sigma=sigma, mass_ratio=mass_ratio, CC=CC)
print(uc.summary())

## 2  Derived scales

The three fundamental ion scales follow directly from $\sigma$ and the mass ratio:

In [ ]:
print(f"Ion inertial length:  di = {uc.di_in_cells:.1f} cells")
print(f"Ion Alfvén speed:     vAi/c = {uc.vAi_over_c:.5f}")
print(f"Ion cyclotron freq:   Ωci/ωpe = {uc.Omega_ci_over_wpe:.5f}")
print()
print(f"Background field:     B0 = {uc.B0:.6f} (code units)")
print(f"Ion E-field scale:    E0 = B0 × vAi/c = {uc.E0:.6f} (code units)")

**Physical interpretation:**
- With `c_omp = 10` and `mass_ratio = 100`, one ion inertial length spans 100 grid cells.
- The ion Alfvén speed is about 3.2% of $c$ — non-relativistic, as expected.
- A time of 1 $\Omega_{ci}^{-1}$ corresponds to roughly $1/\Omega_{ci}/\omega_{pe} \approx 316$ code time steps.

## 3  Converting arrays: length, time, speed

In [ ]:
# --- Length: cells → di ---
x_cells = np.array([0.0, 100.0, 500.0, 1000.0])   # grid positions
x_di    = uc.length(x_cells)

print("x [cells]:", x_cells)
print("x [di]:   ", x_di)
print(f"  (1 di = {uc.di_in_cells:.0f} cells → 100 cells = {uc.length(100.):.1f} di)")
print()

In [ ]:
# --- Time: 1/ωpe → 1/Ωci ---
t_code = np.array([0.0, 1000.0, 5000.0, 10000.0])   # simulation times in 1/ωpe
t_ion  = uc.time(t_code)

print("t [1/ωpe]:", t_code)
print("t [1/Ωci]:", np.round(t_ion, 3))
print(f"  (wpe_to_wci = {uc.wpe_to_wci:.5f})")
print()

In [ ]:
# --- Speed: c → vAi ---
v_c    = np.array([0.0, 0.01, 0.032, 0.1])   # speeds in units of c
v_vAi  = uc.speed(v_c)

print("v [c]:  ", v_c)
print("v [vAi]:", np.round(v_vAi, 3))
print(f"  (1 vAi = {uc.vAi_over_c:.5f} c → 0.032c ≈ 1 vAi: {uc.speed(0.032):.3f})")

## 4  Electromagnetic field normalisation

B fields are stored in Tristan code units where the upstream guide field $B_y \approx B_0$ (a small number like $10^{-3}$–$10^{-2}$).  Dividing by $B_0$ gives dimensionless values where the upstream $B_y \approx 1$.

E fields are normalised to $E_0 = B_0 \times v_{Ai}/c$.  With this convention:
- An ExB drift at exactly $v_{Ai}$ gives `field_E(Ez) / field_B(By) = 1`
- The Poynting flux in ion units is simply `field_E(E) × field_B(B)`

In [ ]:
# Upstream background field has magnitude B0 in code units
B_background_code = uc.B0
print(f"B0 in code units = {uc.B0:.6f}")
print(f"field_B(B0) = {uc.field_B(uc.B0):.4f}   (should be 1.0000)")
print()

# ExB equilibrium: if Bz=0, By=B0, then Ez = -vx * By for drift at vx
v_ExB_c  = 0.01                   # ExB drift speed in units of c
By_code  = uc.B0                  # upstream By ≈ B0 in code units
Ez_code  = -v_ExB_c * By_code     # Ez = -vx * By (code units)

By_ion = uc.field_B(By_code)      # should be 1.0
Ez_ion = uc.field_E(Ez_code)      # should be -vx/vAi

v_ExB_ion = -Ez_ion / By_ion      # = vx / vAi  (ExB = Ez/By in ion units)

print(f"ExB verification:")
print(f"  vx = {v_ExB_c} c = {uc.speed(v_ExB_c):.4f} vAi")
print(f"  field_B(By) = {By_ion:.4f}")
print(f"  field_E(Ez) = {Ez_ion:.4f}")
print(f"  ExB drift = -Ez/By = {v_ExB_ion:.4f} vAi   (should be {uc.speed(v_ExB_c):.4f})")

## 5  Round-trip verification

Converting to ion units and back should recover the original values:

In [ ]:
x_orig = np.array([10.0, 250.0, 800.0])
x_back = uc.length(x_orig) / uc.cell_to_di
print("Length round-trip error:", np.max(np.abs(x_back - x_orig)))

t_orig = np.array([100.0, 5000.0, 12345.0])
t_back = uc.time(t_orig) / uc.wpe_to_wci
print("Time round-trip error:  ", np.max(np.abs(t_back - t_orig)))

v_orig = np.array([0.001, 0.02, 0.1])
v_back = uc.speed(v_orig) / uc.c_to_vAi
print("Speed round-trip error: ", np.max(np.abs(v_back - v_orig)))

## 6  Using unit conversion from a Simulation object

In practice you rarely construct `UnitConverter` by hand — `Simulation.unit_converter` reads the params file automatically:

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────────
SIM_DIR = "/path/to/your/simulation/output"   # <── edit this
# ────────────────────────────────────────────────────────────────────────────

import pytrist

sim = pytrist.Simulation(SIM_DIR)
uc2 = sim.unit_converter
print(uc2.summary())

# Convert coordinate arrays from a particle snapshot in one call:
# sp = sim.particles(step).species(1, units='ion')
# sp['x'] is now in di, sp['u'] is now in vAi

## 7  Visualising the unit scales

A quick sanity plot: how do ion scales compare to electron scales?

In [ ]:
# Sweep mass ratio to show how scales depend on mi/me
mrs  = np.array([1, 4, 16, 25, 100, 400, 1836])
dis  = c_omp * np.sqrt(mrs)                       # di in cells
vais = CC * np.sqrt(sigma / mrs)                  # vAi / c

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].loglog(mrs, dis, "o-")
axes[0].axvline(mass_ratio, ls="--", color="C1", label=f"this run (mr={mass_ratio:.0f})")
axes[0].set_xlabel(r"$m_i / m_e$")
axes[0].set_ylabel(r"$d_i$ [cells]")
axes[0].set_title(r"Ion inertial length vs. mass ratio")
axes[0].legend()
axes[0].grid(True, which="both", alpha=0.3)

axes[1].loglog(mrs, vais / CC, "o-")
axes[1].axvline(mass_ratio, ls="--", color="C1", label=f"this run")
axes[1].set_xlabel(r"$m_i / m_e$")
axes[1].set_ylabel(r"$v_{Ai} / c$")
axes[1].set_title(r"Ion Alfvén speed vs. mass ratio")
axes[1].legend()
axes[1].grid(True, which="both", alpha=0.3)

plt.suptitle(rf"$c_{{omp}} = {c_omp}$,  $\sigma = {sigma}$,  CC $= {CC}$", y=1.01)
plt.tight_layout()
plt.show()